# Week 4 skills notebook: charts that tell the truth

This notebook belongs to the skills track. It uses made-up data about a bus company, a bakery and a runner, so that the skills stay separate from this week's project.

You will take the five-second test, draw bar and line charts that carry the honesty kit, measure how much a cut axis exaggerates, catch an AI chart that misleads, and build a misleading chart on purpose so that you can recognise one.

**How to use it:** run each cell in order. When a cell asks you to write, double-click the text cell and type.

## 1. Three made-up tables

The first cell makes three small tables from a seed, so everyone gets the same numbers.

- The bus table holds the share of buses that arrived on time, each month for three years.
- The bakery table holds the average number of loaves sold per day at five branches.
- The runner table holds one runner's time for 5 km, once a week.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(404)

# A made-up bus company: the share of its buses that arrived on time, each month for three years
t = np.arange(36)
trend = np.where(t < 9, 83.0 + 0.14 * t, 84.26 - 0.09 * (t - 9))
bus = pd.DataFrame({
    "month": pd.date_range("2023-01-01", periods=36, freq="MS"),
    "on_time_pct": (trend + rng.normal(0, 0.3, 36)).round(1),
})

# A made-up bakery: loaves sold per day at five branches, averaged over 30 days
loaves = pd.DataFrame(rng.poisson([71, 72, 74, 73, 75], size=(30, 5)), columns=list("ABCDE"))
bakery = loaves.mean().round(1).rename("loaves_per_day")

# A made-up runner: the time for a 5 km run, once a week for 16 weeks
runner = pd.DataFrame({
    "week": np.arange(1, 17),
    "minutes": (28.4 - 0.13 * np.arange(16) + rng.normal(0, 0.15, 16)).round(2),
})

print(bus.head())
print()
print(bakery)

## 2. The five-second test

A chart acts on you before you think. Run the next cell, look at the chart for five seconds only, and then look away. Do not read the numbers on the axis.

Write down what you now believe about the buses before you read any further.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(bus["month"], bus["on_time_pct"], marker="o", markersize=3)
ax.set_title("Buses on time, by month")
ax.set_xlabel("month")
ax.set_ylabel("buses on time (%)")
default_ax = ax                      # keep the axes, so we can measure them later
plt.tight_layout()
plt.show()

**My first impression (double-click to write):**

**The feature of the chart that pushed me there:**

## 3. The same numbers on an axis from zero

The next chart draws exactly the same numbers. The only change is that the y-axis runs from 0 to 100 percent. Compare it with your first impression.

When code never sets the y-axis, matplotlib starts the axis just below the smallest value. That default makes every change fill the whole height of the chart. The last line of the cell prints where the first chart's axis started.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(bus["month"], bus["on_time_pct"], marker="o", markersize=3)
ax.set_ylim(0, 100)
ax.set_title("Most buses arrive on time, and the share has slipped a little")
ax.set_xlabel("month")
ax.set_ylabel("buses on time (%)")
plt.tight_layout()
plt.show()

print("The first chart's axis starts at:", round(default_ax.get_ylim()[0], 2))
print("This chart's axis starts at:", ax.get_ylim()[0])

## 4. Two ways to say how much

A change between two percentages can be stated in two ways.

- **Percentage points** are the plain difference between the two percentages.
- **Percent** is the change as a share of the starting value.

The cell below works one small example and then measures the bus share from its first month to its last. Always say which of the two you mean.

In [ ]:
before, after = 50, 40
print(f"From {before}% to {after}%: {after - before} points, which is {(after - before) / before * 100:.1f} percent")
print()

first = bus["on_time_pct"].iloc[0]
last = bus["on_time_pct"].iloc[-1]
points = last - first                      # percentage points: the plain difference
percent = (last - first) / first * 100     # percent: the change as a share of where it started
print("First month:", first, "% on time. Last month:", last, "% on time.")
print("Change in percentage points:", round(points, 1))
print("Change in percent:", round(percent, 2))
print("Change per year, in points:", round(points / (len(bus) - 1) * 12, 2))

### Chart the change itself

A line of the level answers the question *how much is there?* When the question is *how fast is it changing?*, chart the change itself.

The method `.diff()` subtracts from each value the value before it. Below, the notebook averages each year, takes the change from one year to the next, and draws the changes as bars from zero. A bar below zero is a fall.

In [ ]:
yearly = bus.groupby(bus["month"].dt.year)["on_time_pct"].mean().round(2)
change = yearly.diff()
print(pd.DataFrame({"yearly average": yearly, "change from the year before": change}))

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(change.index[1:].astype(str), change.iloc[1:])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("The on-time share fell in each of the last two years")
ax.set_xlabel("year")
ax.set_ylabel("change from the year before (points)")
plt.tight_layout()
plt.show()

## 5. Bars compare amounts

A bar shows its amount by its length, so the value axis of a bar chart starts at zero. matplotlib starts bars at zero unless the code says otherwise.

The chart below carries the full honesty kit:

- a title that states the claim, not only the topic;
- axis labels with their units;
- a note that names the source;
- bars that start at zero.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(bakery.index, bakery.values)
ax.set_title("The five branches sell almost the same amount of bread")
ax.set_xlabel("branch")
ax.set_ylabel("loaves sold per day (30-day average)")
ax.text(1.0, -0.3, "Source: made-up data, seed 404", transform=ax.transAxes, ha="right", fontsize=8)
plt.tight_layout()
plt.show()
print("This chart's axis starts at:", ax.get_ylim()[0])

## 6. Measure the exaggeration

The exaggeration factor says how many times bigger a chart makes a change look than it really is.

- The **real change** is measured from zero: the change divided by the starting value.
- The **drawn change** is measured from the bottom of the axis, because that is what the eye sees: the change divided by the starting value's height above the axis bottom.
- The factor is the drawn change divided by the real change. A chart whose axis starts at zero has a factor of 1.

In [ ]:
def exaggeration(base, other, bottom):
    """How many times bigger a chart makes the change from base to other look than it is.
    bottom is where the chart's value axis starts."""
    real = (other - base) / base               # the change as a share of the base value
    drawn = (other - base) / (base - bottom)   # the change as a share of the height drawn above the axis
    return drawn / real

print("A fall from 100 to 95 on an axis that starts at 90: factor", round(exaggeration(100, 95, 90), 1))
print("Real change:", (95 - 100) / 100 * 100, "percent. Drawn change:", (95 - 100) / (100 - 90) * 100, "percent of the height above the axis.")
print("A fall from 50 to 40 on an axis that starts at 30: factor", round(exaggeration(50, 40, 30), 1))
print("Real change:", (40 - 50) / 50 * 100, "percent. Drawn change:", (40 - 50) / (50 - 30) * 100, "percent of the height above the axis.")
print()

bottom = default_ax.get_ylim()[0]
print("Real change, first month to last:", round(percent, 2), "percent")
print("Drawn change in the first chart:", round((last - first) / (first - bottom) * 100, 1), "percent of the starting height")
print("Exaggeration factor of the first chart:", round(exaggeration(first, last, bottom), 1))
print("Exaggeration factor of the chart from zero:", round(exaggeration(first, last, 0), 1))

## 7. When the AI is wrong

Suppose you ask an AI assistant: *"Draw a bar chart of loaves per day for each branch that makes the differences easy to see, and describe what it shows."* It returns the cell below, with this description: *"Branch E sells about four times as much bread as Branch B."*

Run the cell, then decide whether the description is right.

In [ ]:
# Written by an AI assistant
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(bakery.index, bakery.values, color="tab:orange")
ax.set_ylim(bakery.min() - 1, bakery.max() + 1)   # zoom in so the differences are easy to see
ax.set_title("Branch E far outsells Branch B")
ax.set_xlabel("branch")
ax.set_ylabel("loaves per day")
ai_ax = ax
plt.tight_layout()
plt.show()

The chart looks dramatic because the code starts the axis one loaf below the shortest bar. The assistant then described the picture it drew, not the bakery. The AI did what it was asked, and the request itself was the trap: *make the differences easy to see* invites a cut axis.

The next cell measures the bakery and the picture side by side.

In [ ]:
hi, lo = bakery.idxmax(), bakery.idxmin()
bottom = ai_ax.get_ylim()[0]
real_ratio = bakery[hi] / bakery[lo]
drawn_ratio = (bakery[hi] - bottom) / (bakery[lo] - bottom)
print("Tallest bar:", hi, bakery[hi], "loaves. Shortest bar:", lo, bakery[lo], "loaves.")
print("Real ratio of the two branches:", round(real_ratio, 2))
print("Ratio of the bar heights as drawn:", round(drawn_ratio, 1))
print("Exaggeration factor of the AI's chart:", round(exaggeration(bakery[lo], bakery[hi], bottom), 1))
print("The AI's axis runs from", round(ai_ax.get_ylim()[0], 1), "to", round(ai_ax.get_ylim()[1], 1), "loaves")

# CHECK: the AI's "four times" is what the chart draws, not what the bakery sells
assert (hi, lo) == ("E", "B")
assert round(drawn_ratio) == 4
assert real_ratio < 1.1
print("CHECK passed")

## 8. When a zoomed axis is honest

A line chart may start its axis above zero when zero means nothing for the question, or when a small change matters a great deal. The chart must then say so.

Nobody runs 5 km in zero minutes, and a minute is a large gain for a runner. The chart below zooms in and marks the zoom twice: the title states the real change in minutes and where the axis starts, and a note on the chart says that the axis does not start at zero.

Bars never get this exception, because their length is the message.

In [ ]:
start, end = runner["minutes"].iloc[0], runner["minutes"].iloc[-1]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(runner["week"], runner["minutes"], marker="o")
ax.set_ylim(25, 29)
ax.set_title(f"5 km time fell by {start - end:.1f} minutes in 16 weeks (axis starts at 25 minutes)")
ax.set_xlabel("week")
ax.set_ylabel("minutes for 5 km")
ax.text(0.99, 0.92, "axis does not start at zero", transform=ax.transAxes, ha="right", color="firebrick")
plt.tight_layout()
plt.show()
print("First week:", start, "minutes. Last week:", end, "minutes.")
print("Change:", round(end - start, 2), "minutes, which is", round((end - start) / start * 100, 1), "percent")
print("This chart's axis starts at:", ax.get_ylim()[0], "minutes")

## 9. Your turn 1: measure from the best month

Where you start measuring changes the answer. The bus share rose for several months before it fell.

Find the month with the highest on-time share and store it in `best_month`. Then store the change from that month to the last month in `points_from_best` (in percentage points) and `percent_from_best` (in percent). Use your AI assistant if you like, and read its code line by line before you run it.

The CHECK cell also prints the change from the first month, so you can compare the two windows.

In [ ]:
# your turn 1: find the best month, then measure the change from it to the last month
best_month = None          # the month with the highest on-time share
points_from_best = None    # change in percentage points, best month to last month
percent_from_best = None   # change in percent, best month to last month

In [ ]:
# CHECK 1
assert best_month is not None, "best_month is still empty"
top = bus.loc[bus["on_time_pct"].idxmax()]
assert pd.Timestamp(best_month) == top["month"], "best_month is not the month with the highest share"
assert abs(points_from_best - (last - top["on_time_pct"])) < 0.01
assert abs(percent_from_best - (last - top["on_time_pct"]) / top["on_time_pct"] * 100) < 0.01
print("CHECK 1 passed:", top["month"].strftime("%Y-%m"), round(points_from_best, 1), "points,", round(percent_from_best, 1), "percent")
print("From the first month, the change was", round(points, 1), "points.")

## 10. Your turn 2: build the misleading chart on purpose

Making the lie yourself is the fastest way to learn to spot it.

Draw the bus series as a line chart, with the y-axis cut so that the fall from the best month to the last month looks at least 20 times bigger than it is. Keep every point on the chart. Give it a dramatic title, stamp it with the words *misleading on purpose* using `ax.text`, and store its axes in `cut_ax`.

A misleading chart never leaves the notebook without its stamp.

In [ ]:
# your turn 2: a line chart of the bus share with a cut axis and a stamp, stored in cut_ax
cut_ax = None

In [ ]:
# CHECK 2
assert cut_ax is not None, "cut_ax is still empty"
stamp = " ".join(t.get_text().lower() for t in cut_ax.texts)
assert "misleading" in stamp or "misleidend" in stamp, "stamp the chart with the words: misleading on purpose"
assert cut_ax.get_ylim()[0] < bus["on_time_pct"].min(), "keep every point on the chart"
cut_factor = exaggeration(bus["on_time_pct"].max(), last, cut_ax.get_ylim()[0])
assert cut_factor >= 20, f"the factor is only {cut_factor:.1f}; cut the axis closer to the data"
print("CHECK 2 passed: the axis starts at", round(cut_ax.get_ylim()[0], 2), "and the factor is", round(cut_factor, 1))

## 11. Your turn 3: fix the AI's chart

Redraw the bakery chart from section 7 honestly. Keep the same numbers and use vertical bars (`ax.bar`). Start the bars at zero, write a title that states what the chart really shows, label both axes, and store the axes in `fixed_ax`.

Then write one sentence in the text cell below that a bakery manager could trust. Use the real ratio from section 7.

In [ ]:
# your turn 3: an honest bar chart of the bakery, stored in fixed_ax
fixed_ax = None

In [ ]:
# CHECK 3
assert fixed_ax is not None, "fixed_ax is still empty"
assert fixed_ax.get_ylim()[0] == 0, "the bars must start at zero"
assert fixed_ax.get_title().strip() and fixed_ax.get_title() != ai_ax.get_title(), "write a new title that states what the chart shows"
assert fixed_ax.get_ylabel().strip(), "label the y-axis with its unit"
heights = sorted(round(float(p.get_height()), 1) for p in fixed_ax.patches)
assert heights == sorted(bakery.round(1).tolist()), "keep the same numbers as the AI's chart"
print("CHECK 3 passed: the exaggeration factor is now", round(exaggeration(bakery.min(), bakery.max(), fixed_ax.get_ylim()[0]), 1))

**My sentence for the bakery manager (double-click to write):**

## 12. Before you close the notebook

Run the five-second test from section 2 on two people away from the screen. Show each person the first bus chart for five seconds, write down what they believed, and name the feature that pushed them there. Then show them the chart from section 3 and write down what changed.

## 13. Go deeper: the window hunt

This task is harder than the three before it. In section 9 you measured from two starts. Here you measure from every start, to see how much room a chart maker has to choose the story.

1. Store in `window_changes` the change in percentage points from each month to the last month, for every month except the last. Index it by month.
2. Store in `harshest_start` the start month that gives the largest fall, and in `kindest_start` the start month that gives the smallest fall or the largest rise.
3. Store in `n_rises` the number of start months from which the share looks as if it rose.
4. Draw `window_changes` as bars from zero, with a claim title and labelled axes, and store the axes in `window_ax`.

Then write two headlines in the text cell below, one from `harshest_start` and one from `kindest_start`. Both must be true. Write a third sentence that a careful reader could trust.

In [ ]:
# go deeper: every start month, its change to the last month, and the chart of those changes
window_changes = None    # Series indexed by month: change in points from that month to the last month
harshest_start = None    # the start month with the largest fall
kindest_start = None     # the start month with the smallest fall or the largest rise
n_rises = None           # how many start months make the share look as if it rose
window_ax = None

In [ ]:
# CHECK 4
assert window_changes is not None and window_ax is not None, "window_changes or window_ax is still empty"
s = bus.set_index("month")["on_time_pct"]
expected = s.iloc[-1] - s.iloc[:-1]
assert len(window_changes) == len(expected), "one change for every month except the last"
assert (abs(pd.Series(window_changes).values - expected.values) < 0.01).all(), "each value is the last month minus the start month, in points"
assert pd.Timestamp(harshest_start) == expected.idxmin() and pd.Timestamp(kindest_start) == expected.idxmax()
assert n_rises == int((expected > 0).sum())
assert window_ax.get_ylim()[0] <= expected.min() and window_ax.get_ylim()[1] >= 0, "bars from zero: the axis must hold zero and every bar"
print("CHECK 4 passed: harshest start", expected.idxmin().strftime("%Y-%m"), round(expected.min(), 1), "points; kindest start", expected.idxmax().strftime("%Y-%m"), round(expected.max(), 1), "points")
print("Start months that show a rise:", int((expected > 0).sum()), "of", len(expected))
print("Spread between the two headlines:", round(expected.max() - expected.min(), 1), "points")

**Headline from the harshest start (double-click to write):**

**Headline from the kindest start:**

**A sentence a careful reader could trust:**